# 🎯 MNIST CNN Optimization - Session 5 Assignment

## Assignment Objectives

Achieve **>99.4% validation/test accuracy** on MNIST with:
- **<20k Parameters** 
- **<20 Epochs**
- **Batch Normalization** ✅
- **Dropout** ✅ 
- **Global Average Pooling or FC Layer** ✅

## 📋 Architecture Design Principles

This assignment explores key CNN optimization concepts:

1. **Layer Design**: Strategic use of 3x3 convolutions for feature extraction
2. **1x1 Convolutions**: Channel reduction and parameter efficiency
3. **Batch Normalization**: Training stability and faster convergence
4. **Dropout**: Regularization to prevent overfitting (0.1 → 0.15 progressive)
5. **Global Average Pooling**: Replacing heavy FC layers
6. **Receptive Field**: Ensuring adequate coverage for MNIST (28x28)
7. **MaxPooling Placement**: Strategic spatial downsampling
8. **Learning Rate Scheduling**: StepLR for better convergence
9. **Early Stopping**: Preventing overfitting

## 🏗️ Model Architectures

### 1. TinyNet (~1.4k params)
- Ultra-lightweight baseline
- Single pooling operation
- GAP + minimal FC

### 2. BetterTinyNet (~18.9k params) 
- **TARGET MODEL** for assignment
- Three progressive blocks
- Strategic 1x1 channel reduction
- No FC layer (GAP only)

### 3. ElegantOptimizedNet (~20k params)
- Advanced residual connections
- Optimal gradient flow
- Highest accuracy potential


In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
from datetime import datetime

# Import our custom modules
from models import TinyNet, BetterTinyNet, ElegantOptimizedNet, get_model_summary
from utils import get_data_loaders, train_model, plot_training_history, get_device

# Set up device
device = get_device()
print(f"Device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


In [ ]:
# Load MNIST dataset
print("Loading MNIST dataset...")
train_loader, test_loader = get_data_loaders(batch_size=64, test_batch_size=1000)

# Create model instances and show summaries
models = {
    'TinyNet': TinyNet(),
    'BetterTinyNet': BetterTinyNet(),
    'ElegantOptimizedNet': ElegantOptimizedNet()
}

print("\n" + "="*60)
print("MODEL SUMMARIES")
print("="*60)

for name, model in models.items():
    summary = get_model_summary(model, name)
    print(f"\n{name}:")
    print(f"  Parameters: {summary['total_parameters']:,}")
    print(f"  Efficiency: {summary['parameter_efficiency']}")
    print(f"  Memory: {summary['memory_footprint']}")
    
    # Check assignment requirements
    meets_param_req = summary['total_parameters'] < 20000
    print(f"  <20k params: {'✅' if meets_param_req else '❌'}")

print("\n" + "="*60)


## 🚀 Training BetterTinyNet (Target Model)

This is our **primary target model** designed to achieve >99.4% accuracy with <20k parameters.


In [ ]:
# Train BetterTinyNet - Our target model
print("🎯 Training BetterTinyNet - Target Model")
print("="*50)

better_tiny_net = BetterTinyNet()
history_better = train_model(
    model=better_tiny_net,
    model_name="BetterTinyNet",
    device=device,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=20,
    lr=0.01,
    target_accuracy=99.4
)

# Display results
print(f"\n🏆 BetterTinyNet Results:")
print(f"   Best Accuracy: {history_better['best_accuracy']:.2f}%")
print(f"   Parameters: {history_better['total_params']:,}")
print(f"   Target Achieved: {'✅' if history_better['target_achieved'] else '❌'}")
print(f"   Training Time: {history_better['total_time']:.1f}s")


## 📊 Comparative Analysis - All Models

Let's train all three models to compare their performance and parameter efficiency.


In [ ]:
# Train all models for comparison
print("🔄 Training All Models for Comparison")
print("="*60)

all_histories = {}

# If BetterTinyNet was already trained above, use that history
if 'history_better' in locals():
    all_histories['BetterTinyNet'] = history_better

# Train remaining models
models_to_train = {
    'TinyNet': TinyNet(),
    'ElegantOptimizedNet': ElegantOptimizedNet()
}

for model_name, model in models_to_train.items():
    print(f"\n🚀 Training {model_name}...")
    
    history = train_model(
        model=model,
        model_name=model_name,
        device=device,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=20,
        lr=0.01,
        target_accuracy=99.4
    )
    
    all_histories[model_name] = history
    
    print(f"\n📈 {model_name} Results:")
    print(f"   Best Accuracy: {history['best_accuracy']:.2f}%")
    print(f"   Parameters: {history['total_params']:,}")
    print(f"   Target Achieved: {'✅' if history['target_achieved'] else '❌'}")

print("\n" + "="*60)
print("✅ All Models Trained!")
print("="*60)


In [ ]:
# Generate visualization of training results
print("📊 Generating Training Visualizations...")

plot_training_history(all_histories)

# Generate summary table
from utils import generate_summary_table

print("\n" + "="*80)
print("📋 FINAL RESULTS SUMMARY")
print("="*80)

summary_table = generate_summary_table(all_histories)
print(summary_table)


In [ ]:
# Assignment Requirements Verification
print("\n🎯 ASSIGNMENT REQUIREMENTS VERIFICATION")
print("="*80)

requirements_met = {
    'accuracy': False,
    'parameters': False,
    'epochs': False,
    'batch_norm': True,  # All models use BN
    'dropout': True,     # All models use dropout
    'gap_or_fc': True    # All models use GAP
}

print("\nDetailed Check for Each Model:")
print("-" * 40)

for model_name, history in all_histories.items():
    print(f"\n{model_name}:")
    
    # Check individual requirements
    params_ok = history['total_params'] < 20000
    accuracy_ok = history['best_accuracy'] >= 99.4
    epochs_ok = len(history['train_acc']) <= 20
    
    print(f"   ✅ Test Accuracy: {history['best_accuracy']:.2f}% {'>= 99.4%' if accuracy_ok else '< 99.4% ❌'}")
    print(f"   ✅ Parameters: {history['total_params']:,} {'< 20k' if params_ok else '> 20k ❌'}")
    print(f"   ✅ Epochs Used: {len(history['train_acc'])} <= 20")
    print(f"   ✅ Batch Normalization: Used in all layers")
    print(f"   ✅ Dropout: Progressive rates (0.1 → 0.15)")
    print(f"   ✅ Global Average Pooling: Replaces FC layers")
    
    # Overall check
    all_met = params_ok and accuracy_ok and epochs_ok
    print(f"   {'🎉 ALL REQUIREMENTS MET!' if all_met else '❌ Some requirements not met'}")
    
    # Update global requirements
    if all_met:
        requirements_met['accuracy'] = accuracy_ok
        requirements_met['parameters'] = params_ok
        requirements_met['epochs'] = epochs_ok

print("\n" + "="*80)
print("🏆 ASSIGNMENT SUCCESS STATUS")
print("="*80)

success_models = [name for name, history in all_histories.items() 
                  if history['best_accuracy'] >= 99.4 and history['total_params'] < 20000]

if success_models:
    print(f"✅ SUCCESS! Models meeting all requirements: {', '.join(success_models)}")
    best_model = max(success_models, key=lambda x: all_histories[x]['best_accuracy'])
    print(f"🏆 Best performing model: {best_model} ({all_histories[best_model]['best_accuracy']:.2f}%)")
else:
    print("❌ No models met all requirements. Need further optimization.")

print(f"\nTimestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)
